# IMPORTING

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# relative to the notebooks folder
DATA_FOLDER = Path("../.csv/2025-oct to 2026-oct")

def load(name):
    df = pd.read_csv(DATA_FOLDER / f"{name}.csv", dtype_backend="numpy_nullable")
    for col in df.columns:
        if col.endswith("_at"):
            df[col] = pd.to_datetime(df[col], utc=True)
    return df

prs = load("pull_requests")
issues = load("issues")
reviews = load("reviews")
review_comments = load("review_comments")
comments = load("comments")
users = load("users")

items = pd.concat(
    [prs.assign(type="pull_request"), issues.assign(type="issue")],
    ignore_index=True,
)

all_dfs = [items, prs, issues, reviews, review_comments, comments, users]

meta = json.loads((DATA_FOLDER / "dataset.json").read_text())
meta

{'owner': 'godotengine',
 'repo': 'godot',
 'from': '2025-10-18T00:00:00+00:00',
 'to': '2026-10-18T00:00:00+00:00',
 'sample_size': None,
 'seed': 42,
 'include_issues': True,
 'include_pull_requests': True,
 'fetch_user_profiles': True,
 'exported_at': '2026-10-23T16:03:52.5563767+00:00'}

# BOT FILTERING

In [2]:
# from dotenv import load_dotenv
# import os

# load_dotenv()
# ACCESS_TOKEN = os.getenv("ACCESS_TOKEN")

In [3]:
# from rabbit_ng import run_rabbit

# for result in run_rabbit(
#     contributors=list(users["login"]),
#     api_key=ACCESS_TOKEN,
#     min_events=5,
#     min_confidence=1.0,
#     max_queries=3,
#     no_wait=False,
# ):
#     if result.user_type == "Bot":
#         print(f"{result.contributor}: (Confidence: {result.confidence})")

In [4]:
EXTRA_BOTS = set()  # accounts that appear like bot accounts after manual inspection

def is_bot(login, account_type) -> pd.Series:
    return (
        (account_type == "Bot")
        | login.str.endswith("[bot]", na=False)
        | login.isin(EXTRA_BOTS)
    )

BOTS = set()

for df in all_dfs:  # update all df-s with an is_bot field and populates the BOTS set
    login_col = "reviewer" if df is reviews else "login" if df is users else"author"
    df["is_bot"] = is_bot(df[login_col], df["type" if df is users else f"{login_col}_type"])
    BOTS.update(df.loc[df["is_bot"], login_col].dropna())

BOTS

{'Copilot', 'copilot-pull-request-reviewer[bot]', 'dependabot[bot]'}

### Example for bot
- Copilot-pull-request-reviewer[bot]
- dependabot[bot]

### Counterexample

# ALIAS FILTERING

### Normalising the data

In [ ]:
import re
import unicodedata

def normalize_name(name: str):
    s = unicodedata.normalize("NFKD", name)
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()  # José -> jose
    tokens = re.sub(r"[\W\d_]+", " ", s).split()                       # drop punctuation and digits
    return " ".join(sorted(tokens))                                    # word order doesn't matter

def normalize_login(login: str):
    s = re.sub(r"[-_.]", "", login.lower())  # drop separators
    return re.sub(r"\d+$", "", s) or s       # drop trailing digits

def normalize_link(link: str):
    s = re.sub(r"^https?://", "", link.strip().lower())
    return re.sub(r"^www\.", "", s).rstrip("/")

norm_users = pd.DataFrame()

norm_users["login"] = users[~users["is_bot"]]["login"]
norm_users["norm_login"] = users["login"].map(normalize_login)
norm_users["norm_name"] = users["name"].fillna("").map(normalize_name)
norm_users["norm_email"] = users["email"].fillna("").str.strip().str.lower()
norm_users["norm_blog"] = users["blog"].fillna("").map(normalize_link)

norm_users

,login,norm_login,norm_name,norm_email,norm_blog
0,014-code,014code,林桉,,014-code.github.io/014.github.io
1,0MAD03,0mad,,,
2,0TheJCM0,0thejcm,thejcm,,
3,0ttoman,0ttoman,,,
4,0x0ACB,0x0acb,acb x,,
...,...,...,...,...,...
4480,Zybyte85,zybyte,caden stephens,,
4481,Zylann,zylann,marc,,zylannprods.fr/fr
4482,zzag,zzag,vlad zahorodnii,contact@vladzahorodnii.com,blog.vladzahorodnii.com
4483,zzimbaa,zzimbaa,albin blomqvist,,


### Defining filtering methods

In [6]:
from difflib import SequenceMatcher

EXTRA_ALIASES = {}

FILTER_LOGIN = False
FILTER_DISPLAY = False
FILTER_BLOG = False
FILTER_EMAIL = False

NAME_THRESHOLD = 0.9  # 0..1, how alike two names/logins must be
MIN_LENGTH = 4         # ignore very short names/logins, they match too easily

def is_similar(x: str, y: str):
    if len(x) < MIN_LENGTH or len(y) < MIN_LENGTH:
        return False
    m = SequenceMatcher(None, x, y)
    # the two quick ratios are cheap upper bounds that rule out most pairs before the slow exact ratio
    return m.real_quick_ratio() >= NAME_THRESHOLD and m.quick_ratio() >= NAME_THRESHOLD and m.ratio() >= NAME_THRESHOLD

# the 2 measures
def similar_name(a, b):
    return (FILTER_LOGIN and is_similar(a.norm_login, b.norm_login)) or (FILTER_DISPLAY and is_similar(a.norm_name, b.norm_name))

def same_contact(a, b):
    def match(x, y) -> bool: 
        return x != "" and x == y
    return (FILTER_BLOG and match(a.norm_blog, b.norm_blog)) or (FILTER_EMAIL and match(a.norm_email, b.norm_email))

# wrapper for the 2 measures
def is_alias(a, b):
    return similar_name(a, b) or same_contact(a, b)

### Filtering the data

In [7]:
from itertools import combinations

ALIASES = EXTRA_ALIASES.copy()

def add_alias(a, b):
    main_a = ALIASES.get(a, a)
    main_b = ALIASES.get(b, b)
    if main_a == main_b: return  # already the same person

    # everything that pointed to main_a now points to main_b, then main_a itself does too
    for alt, main in ALIASES.items():
        if main == main_a:
            ALIASES[alt] = main_b
    ALIASES[main_a] = main_b

# populate ALIASES
for a, b in combinations(norm_users.itertuples(index=False), 2):
    if is_alias(a, b): add_alias(a.login, b.login)

# make new column for the actual main account of the author
for df in all_dfs:
    login_col = "reviewer" if df is reviews else "login" if df is users else "author"
    df["main_login"] = df[login_col].replace(ALIASES)

# make a new column for reviews/comments that describes if the review/comment was authored by the thing it is reviewing/commenting on
author_of = items.set_index("number")["main_login"]
reviews["pr_author"] = reviews["pr_number"].map(author_of)
reviews["by_author"] = (reviews["main_login"] == reviews["pr_author"]).fillna(False)
comments["item_author"] = comments["parent_number"].map(author_of)
comments["by_author"] = (comments["main_login"] == comments["item_author"]).fillna(False)
review_comments["pr_author"] = review_comments["pr_number"].map(author_of)
review_comments["by_author"] = (review_comments["main_login"] == review_comments["pr_author"]).fillna(False)

ALIASES

{}

### Example for alias
- UweSchneiderGmbH : UweSchneider

### Counterexample
- kisg : konczg

# PROCESS-METRICS

## 1. Time to First Response: [6] 
A measurement of how long it takes for another person to respond to an action that requires attention (Like a pull request or issue).

### Applying filters to data

In [ ]:
# exclude items that were authored by bots
m1_items = items[~items["is_bot"]].copy()

# exculde reviews/comments that were authored by bots
m1_reviews = reviews[~reviews["is_bot"]].copy()
m1_comments = comments[~comments["is_bot"]].copy()
m1_review_comments = review_comments[~review_comments["is_bot"]].copy()

# exclude reviews/comments that were created by the author of their relevant items
m1_reviews = m1_reviews[~m1_reviews["by_author"]]
m1_comments = m1_comments[~m1_comments["by_author"]]
m1_review_comments = m1_review_comments[~m1_review_comments["by_author"]]

### Calculating metric

In [ ]:

m1_items["first_review"] = m1_items["number"].map(m1_reviews.groupby("pr_number")["submitted_at"].min()) - m1_items["created_at"]
m1_items["first_comment"] = m1_items["number"].map(m1_comments.groupby("parent_number")["created_at"].min()) - m1_items["created_at"]
m1_items["first_review_comment"] = m1_items["number"].map(m1_review_comments.groupby("pr_number")["created_at"].min()) - m1_items["created_at"]

m1_items["first_response"] = m1_items[["first_review", "first_comment", "first_review_comment"]].min(axis=1)

m1_items

,number,title,state,author,author_type,created_at,updated_at,closed_at,merged_at,merged,...,state_reason,closed_by,assignees,locked,is_bot,main_login,first_review_time,first_comment_time,first_review_comment_time,first_response
0,111780,Allow double-clicking icons of non-editable Tr...,closed,aaronfranke,User,2025-10-18 02:47:42+00:00,2026-02-07 12:35:53+00:00,2025-11-10 23:52:49+00:00,2025-11-10 23:52:49+00:00,True,...,<NA>,<NA>,<NA>,<NA>,False,aaronfranke,23 days 18:03:17,23 days 18:03:08,NaT,23 days 18:03:08
1,111782,Update note for `rendering/rendering_device/vs...,closed,Kaleb-Reid,User,2025-10-18 04:40:00+00:00,2026-01-25 09:39:53+00:00,2025-10-20 23:15:21+00:00,2025-10-20 23:15:21+00:00,True,...,<NA>,<NA>,<NA>,<NA>,False,Kaleb-Reid,0 days 00:53:26,2 days 18:35:48,0 days 03:09:43,0 days 00:53:26
2,111783,Implement Directional Shadow Splits debug mode...,open,Kaleb-Reid,User,2025-10-18 07:13:08+00:00,2026-06-19 06:55:50+00:00,NaT,NaT,False,...,<NA>,<NA>,<NA>,<NA>,False,Kaleb-Reid,0 days 14:39:32,NaT,NaT,0 days 14:39:32
3,111784,Add Unit Tests for StreamPeerTLS,open,Sherv01,User,2025-10-18 07:38:50+00:00,2026-02-17 20:10:07+00:00,NaT,NaT,False,...,<NA>,<NA>,<NA>,<NA>,False,Sherv01,NaT,0 days 00:25:34,NaT,0 days 00:25:34
4,111785,Add C# Bindings for GDExtension types and corr...,open,Ryan-000,User,2025-10-18 09:17:59+00:00,2026-02-17 20:10:06+00:00,NaT,NaT,False,...,<NA>,<NA>,<NA>,<NA>,False,Ryan-000,NaT,NaT,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12742,124587,C#: released weak GC handle left after a dead-...,open,begezavr2,User,2026-10-17 18:54:54+00:00,2026-10-19 09:13:05+00:00,NaT,NaT,<NA>,...,<NA>,<NA>,<NA>,False,False,begezavr2,NaT,1 days 14:18:02,NaT,1 days 14:18:02
12743,124588,extends Node func _ready(): # Enable V-Sy...,closed,mdosman01833759281-commits,User,2026-10-17 19:22:19+00:00,2026-10-17 19:35:22+00:00,2026-10-17 19:34:25+00:00,NaT,<NA>,...,not_planned,rsubtil,<NA>,False,False,mdosman01833759281-commits,NaT,NaT,NaT,NaT
12744,124591,"Sticky items hidden while dragging, but still ...",open,AeioMuch,User,2026-10-17 21:47:11+00:00,2026-10-19 08:43:02+00:00,NaT,NaT,<NA>,...,<NA>,<NA>,<NA>,False,False,AeioMuch,NaT,NaT,NaT,NaT
12745,124592,Godot refuses to use OpenGL if driver support ...,open,jummy0,User,2026-10-17 22:12:17+00:00,2026-10-19 09:19:36+00:00,NaT,NaT,<NA>,...,<NA>,<NA>,<NA>,False,False,jummy0,NaT,0 days 00:06:56,NaT,0 days 00:06:56


### Reporting Data